
Instalação das bibliotecas necessárias

---


*   Langchain > Orquestrador: Encadeia etapas de processamento, como receber uma pergunta, buscar dados, montar o contexto e gerar a resposta.

*   ChromaDB > anco de dados vetorial de código aberto (licença Apache 2.0) criado para armazenar, gerenciar e consultar embeddings

*   pypdf > Biblioteca para ler, mapnipular e modificar arquivos PDF





In [ ]:
# Versões fixadas para o notebook rodar igual em qualquer máquina.
# (langchain 0.1.x mudou bastante nas versões seguintes; sem fixar, os imports quebram)

# Orquestração (LangChain)
!pip install -q langchain==0.1.20 langchain-community==0.0.38

# Conexão com o LLM hospedado no Groq
!pip install -q langchain-groq==0.1.3

# Banco vetorial
!pip install -q chromadb==0.5.0

# Leitura do PDF
!pip install -q pypdf==4.2.0

# Modelo de embeddings (HuggingFace)
!pip install -q sentence-transformers==2.7.0

In [ ]:
# Importações básicas
import os

# loader de documentos PDF
from langchain_community.document_loaders import PyPDFLoader

# Divisão de texto em blocos
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings
from langchain_openai import OpenAIEmbeddings

# Banco vetorial
from langchain_community.vectorstores import Chroma

# LLM
from langchain_openai import ChatOpenAI

# Cadeia RAG
from langchain.chains import RetrievalQA

Lendo o arquivo "Analise de mercado" em PDF

In [ ]:
# Caminho do PDF com a análise de mercado
CAMINHO_PDF = "/content/Analise de mercado 1° TRI ABF.pdf"

# Carrega o PDF
loader = PyPDFLoader(CAMINHO_PDF)
documents = loader.load()

# Quantidade de páginas carregadas
len(documents)

23

Criando os vetores, transformando o texto do PDF em CHUNKS

(pedações de texto que estarão na Vector Store)

In [ ]:
# Divide os documentos em chunks menores
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

len(chunks)

31

Criando a camada de vetores numéricos e banco vetorial

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Inicializa embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# Cria o banco vetorial
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_abf_franchising"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Recuperação dos 3 argumentos mais próximos do PROMPT

In [ ]:
# Cria o retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

In [ ]:
#Criando conexão com o modelo LLAMA hospedado no GROQ, "Temperatura = 0 deixa mais determinístico"
from google.colab import userdata
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    groq_api_key=userdata.get("GROQ_API_KEY"),
    temperature=0
)

# Cria a cadeia RAG
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)

Melhorando a qualidade das respostas, pois as inmstruções não dizem ao modelo para se limitar apenas ao contexto, fazendo com que ele responda fora da janela de informações.

In [ ]:
from langchain.prompts import PromptTemplate

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""Responda em português, usando apenas o contexto abaixo.
Se a resposta não estiver no contexto, diga que não encontrou essa informação no documento.

Contexto:
{context}

Pergunta: {question}

Resposta:"""
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": prompt}
)

In [ ]:
resposta = qa_chain.invoke({"query": "Qual segmento teve maior crescimento e de quanto?"})

print(resposta["result"])

for doc in resposta["source_documents"]:
    print("Página:", doc.metadata.get("page"))

O segmento que apresentou o maior crescimento foi **Alimentação – Comercialização e Distribuição**, com um aumento de **22,0 %** no faturamento entre o 1T 2026 e o 1T 2025.
Página: 15
Página: 15
Página: 11
